# Individual Porfolio
-------------------------------------------------------------------------------------------------------------------------------------------------------------

## Basketball Game
-------------------------------------------------------------------------------------------------------------------------------------------------------------

## Basketball Evasion — CS111 Code Review

I built a basketball evasion game where you dodge LeBron James on a court. Below I break down every CS111 concept using real code from the game.

---

## Control Structures

### Iteration — `for`, `forEach`, `while` loops

The court's wood plank lines use a `for` loop:
```javascript
for (let x = 0; x < W; x += W/28) {
    ctx.beginPath(); ctx.moveTo(x, 0); ctx.lineTo(x, H); ctx.stroke();
}
```

The bench legs use `forEach` to draw 3 positions without repeating code:
```javascript
[.1, .5, .82].forEach(fx => 
    ctx.fillRect(o.x + o.w*fx, o.y + o.h*.4, o.w*.1, o.h*.55)
);
```

---

### Conditionals — `if/else`

Only move if the path isn't blocked:
```javascript
if (!blocked(p.x + dx, p.y, PR)) p.x += dx;
if (!blocked(p.x, p.y + dy, PR)) p.y += dy;
```

LeBron tries direct path first, then slides along walls:
```javascript
if (!blocked(l.x+mx, l.y+my, LR)) { l.x += mx; l.y += my; }
else {
    if (!blocked(l.x+mx, l.y, LR)) l.x += mx;
    if (!blocked(l.x, l.y+my, LR)) l.y += my;
}
```

---

### Nested Conditions — multi-level conditionals

Catch check with nested high score update:
```javascript
if (dist < PR + LR + 2) {
    over = true;
    if (elapsed > best) best = elapsed;
}
```

Speed warning with nested opacity calculation:
```javascript
if (spd > 2.8) {
    const alpha = Math.min((spd - 2.8) * 0.5, 0.9);
    ctx.fillStyle = `rgba(230,50,50,${alpha})`;
    ctx.fillText('⚡ LeBron is heating up!', W/2, 54);
}
```

---

## Data Types

### Numbers — position, velocity, score tracking
```javascript
const PR = 12;    // player radius
const LR = 15;    // LeBron radius
let elapsed = 0;  // time survived
let best = 0;     // best time
let tick = 0;     // frame counter
let dx = 0, dy = 0;  // velocity
```

---

### Strings — character names, game states, template literals
```javascript
ctx.fillText('11', 0, -28);   // player jersey number string
ctx.fillText('23', 0, -28);   // LeBron jersey number string
ctx.fillText(`⏱ ${elapsed.toFixed(1)}s`, W/2, 24);  // template literal
keys[e.key.toLowerCase()] = true;  // string key name
```

---

### Booleans — flags like `isOver`, `isPaused`
```javascript
let over = false;           // boolean — is game over?
if (over) return;           // gate all logic
if (elapsed > best) best = elapsed;
if (e.key.toLowerCase() === 'r' && over) reset();  // && AND
```

---

### Arrays — game object collections, level data
```javascript
const OBS = [
    { x: W*.25, y: H*.25, w: W*.12, h: H*.08 },
    { x: W*.55, y: H*.55, w: W*.12, h: H*.08 },
    // boundary walls...
];
[.1, .5, .82].forEach(fx => ctx.fillRect(...));
['arrowup','arrowdown','arrowleft','arrowright'].includes(e.key.toLowerCase())
```

---

### Objects (JSON) — configuration objects, sprite data
```javascript
let p = { x: W*.08, y: H*.5, dir: 1 };   // player object
let l = { x: W*.90, y: H*.5, dir: -1 };  // LeBron object
let keys = {};  // keyboard state map
{ x: W*.25, y: H*.25, w: W*.12, h: H*.08 }  // obstacle config object
```

---

## Operators

### Mathematical — physics calculations
```javascript
const dist = Math.sqrt(ddx*ddx + ddy*ddy);  // distance formula
const nx = ddx / dist;   // normalize direction
const mx = nx * spd;     // scale by speed
if (dx && dy) { dx *= 0.707; dy *= 0.707; }  // diagonal fix
const spd = Math.min(1.5 + elapsed * 0.04, 4);  // speed over time
const bounce = Math.sin(tick * 0.22) * 5;  // sine wave bounce
```

---

### String Operations — template literals, concatenation
```javascript
ctx.fillText(`⏱ ${elapsed.toFixed(1)}s`, W/2, 24);
ctx.fillText(`Best: ${best.toFixed(1)}s`, W/2+145, 24);
ctx.fillText(`Survived  ${elapsed.toFixed(1)}s`, W/2, by+105);
keys[e.key.toLowerCase()] = true;  // .toLowerCase() string method
```

---

### Boolean Expressions — `&&`, `||`, `!`
```javascript
if (keys['w'] || keys['arrowup']) dy = -3.5;   // OR
if (keys['a'] || keys['arrowleft']) dx = -3.5;  // OR
if (!blocked(p.x+dx, p.y, PR)) p.x += dx;       // NOT
if (e.key.toLowerCase() === 'r' && over) reset(); // AND
if (dx && dy) { dx *= .707; dy *= .707; }         // AND
```

---

## Play the Game

In [ ]:
%%js
// GAME_RUNNER: Just the court! See the basketball court background with benches.

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';

class GameLevelBasketball {
    constructor(gameEnv) {
        this.classes = [];
        const W = gameEnv.innerWidth  || window.innerWidth;
        const H = gameEnv.innerHeight || window.innerHeight;
        setTimeout(() => this._boot(W, H), 200);
    }

    _boot(W, H) {
        const old = document.getElementById('bball-game');
        if (old) old.remove();

        const canvas = document.createElement('canvas');
        canvas.id = 'bball-game';
        canvas.width = W; canvas.height = H;
        canvas.style.cssText = 'display:block;position:fixed;top:0;left:0;width:100vw;height:100vh;z-index:9999;outline:none;';
        document.body.appendChild(canvas);
        const ctx = canvas.getContext('2d');

        const OBS = [
            { x: W*.25, y: H*.25, w: W*.12, h: H*.08 },
            { x: W*.55, y: H*.55, w: W*.12, h: H*.08 },
            { x: W*.35, y: H*.60, w: W*.10, h: H*.08 },
            { x: W*.60, y: H*.20, w: W*.10, h: H*.08 },
        ];

        const drawCourt = () => {
            const g = ctx.createLinearGradient(0,0,W,0);
            g.addColorStop(0, '#b5651d'); g.addColorStop(.5, '#cd853f'); g.addColorStop(1, '#b5651d');
            ctx.fillStyle = g; ctx.fillRect(0,0,W,H);
            ctx.strokeStyle = 'rgba(90,40,0,.2)'; ctx.lineWidth = 1;
            for (let x = 0; x < W; x += W/28) { ctx.beginPath(); ctx.moveTo(x,0); ctx.lineTo(x,H); ctx.stroke(); }
            ctx.strokeStyle = 'rgba(255,255,255,.7)'; ctx.lineWidth = 2;
            const pad = 20;
            ctx.strokeRect(pad, pad, W-pad*2, H-pad*2);
            ctx.beginPath(); ctx.moveTo(W/2,pad); ctx.lineTo(W/2,H-pad); ctx.stroke();
            ctx.beginPath(); ctx.arc(W/2, H/2, H*.13, 0, Math.PI*2); ctx.stroke();
        };

        const drawObs = () => OBS.forEach(o => {
            ctx.fillStyle = '#7a5230'; ctx.fillRect(o.x, o.y, o.w, o.h*.4);
            ctx.fillStyle = '#4a2f10';
            [.1,.5,.82].forEach(fx => ctx.fillRect(o.x+o.w*fx, o.y+o.h*.4, o.w*.1, o.h*.55));
        });

        const render = () => {
            ctx.clearRect(0,0,W,H);
            drawCourt();
            drawObs();
        };

        render(); // Draw once, no game loop needed
    }
}

export const gameLevelClasses = [GameLevelBasketball];
export { GameControl };

%%html
<div style="width:100%;background:#111;border-radius:8px;overflow:hidden;margin:20px 0;">
<canvas id="bball-cs111" style="display:block;width:100%;"></canvas>
<p style="font-family:monospace;font-size:12px;color:#888;padding:8px 12px;margin:0;background:#1a1a1a;">
Click court to focus · WASD / Arrow Keys to move · R to restart
</p>
</div>
<script>
(function(){
const canvas=document.getElementById('bball-cs111');
if(!canvas)return;
const W=canvas.width=canvas.offsetWidth||700;
const H=canvas.height=Math.round(W*.52);
canvas.style.height=H+'px';
canvas.setAttribute('tabindex','0');
const ctx=canvas.getContext('2d');
const OBS=[
  {x:W*.25,y:H*.25,w:W*.12,h:H*.08},{x:W*.55,y:H*.55,w:W*.12,h:H*.08},
  {x:W*.35,y:H*.60,w:W*.10,h:H*.08},{x:W*.60,y:H*.20,w:W*.10,h:H*.08},
  {x:0,y:0,w:4,h:H},{x:W-4,y:0,w:4,h:H},{x:0,y:0,w:W,h:4},{x:0,y:H-4,w:W,h:4},
];
const PR=12,LR=15;
let p,l,keys,over,t0,elapsed,best=0,tick;
const reset=()=>{
  p={x:W*.08,y:H*.5,dir:1};l={x:W*.90,y:H*.5,dir:-1};
  keys={};over=false;t0=Date.now();elapsed=0;tick=0;
};
reset();
canvas.addEventListener('keydown',e=>{
  keys[e.key.toLowerCase()]=true;
  if(['arrowup','arrowdown','arrowleft','arrowright'].includes(e.key.toLowerCase()))e.preventDefault();
});
canvas.addEventListener('keyup',e=>{
  keys[e.key.toLowerCase()]=false;
  if(e.key.toLowerCase()==='r'&&over)reset();
});
canvas.addEventListener('click',()=>canvas.focus());
const blocked=(cx,cy,r)=>OBS.some(o=>{
  const nx=Math.max(o.x,Math.min(cx,o.x+o.w));
  const ny=Math.max(o.y,Math.min(cy,o.y+o.h));
  return (cx-nx)**2+(cy-ny)**2<r*r;
});
const drawCourt=()=>{
  const g=ctx.createLinearGradient(0,0,W,0);
  g.addColorStop(0,'#b5651d');g.addColorStop(.5,'#cd853f');g.addColorStop(1,'#b5651d');
  ctx.fillStyle=g;ctx.fillRect(0,0,W,H);
  ctx.strokeStyle='rgba(90,40,0,.2)';ctx.lineWidth=1;
  for(let x=0;x<W;x+=W/28){ctx.beginPath();ctx.moveTo(x,0);ctx.lineTo(x,H);ctx.stroke();}
  ctx.strokeStyle='rgba(255,255,255,.7)';ctx.lineWidth=2;
  const pad=16;
  ctx.strokeRect(pad,pad,W-pad*2,H-pad*2);
  ctx.beginPath();ctx.moveTo(W/2,pad);ctx.lineTo(W/2,H-pad);ctx.stroke();
  ctx.beginPath();ctx.arc(W/2,H/2,H*.13,0,Math.PI*2);ctx.stroke();
};
const drawObs=()=>OBS.slice(0,4).forEach(o=>{
  ctx.fillStyle='#7a5230';ctx.fillRect(o.x,o.y,o.w,o.h*.4);
  ctx.fillStyle='#4a2f10';
  [.1,.5,.82].forEach(fx=>ctx.fillRect(o.x+o.w*fx,o.y+o.h*.4,o.w*.1,o.h*.55));
});
const drawFigure=(x,y,jersey,num,dir)=>{
  ctx.save();ctx.translate(Math.round(x),Math.round(y));
  ctx.fillStyle='rgba(0,0,0,.22)';
  ctx.beginPath();ctx.ellipse(0,4,10,3,0,0,Math.PI*2);ctx.fill();
  ctx.fillStyle='#111';ctx.fillRect(-9,-5,8,5);ctx.fillRect(1,-5,8,5);
  ctx.fillStyle=jersey;ctx.fillRect(-9,-20,18,15);ctx.fillRect(-10,-38,20,18);
  ctx.fillStyle='#c8784a';ctx.fillRect(-15,-37,5,13);ctx.fillRect(10,-37,5,13);
  ctx.fillStyle='#fff';ctx.font='bold 9px monospace';
  ctx.textAlign='center';ctx.textBaseline='middle';ctx.fillText(num,0,-28);
  ctx.fillStyle='#c8784a';ctx.fillRect(-3,-41,6,4);
  ctx.beginPath();ctx.arc(0,-50,9,0,Math.PI*2);ctx.fill();
  ctx.fillStyle='#111';ctx.fillRect(dir*2,-52,3,3);
  ctx.restore();
};
const drawBall=(x,y)=>{
  const b=Math.sin(tick*.22)*5;
  ctx.save();ctx.translate(Math.round(x),Math.round(y+b));
  ctx.fillStyle='rgba(0,0,0,.2)';
  ctx.beginPath();ctx.ellipse(0,14-b*.4,8,2.5,0,0,Math.PI*2);ctx.fill();
  ctx.fillStyle='#e65100';ctx.beginPath();ctx.arc(0,0,9,0,Math.PI*2);ctx.fill();
  ctx.strokeStyle='#111';ctx.lineWidth=1;
  ctx.beginPath();ctx.moveTo(-9,0);ctx.lineTo(9,0);ctx.stroke();
  ctx.beginPath();ctx.arc(0,0,9,0,Math.PI);ctx.stroke();
  ctx.restore();
};
const update=()=>{
  if(over)return;
  elapsed=(Date.now()-t0)/1000;tick++;
  let dx=0,dy=0;
  if(keys['w']||keys['arrowup'])dy=-3.5;
  if(keys['s']||keys['arrowdown'])dy=3.5;
  if(keys['a']||keys['arrowleft'])dx=-3.5;
  if(keys['d']||keys['arrowright'])dx=3.5;
  if(dx&&dy){dx*=.707;dy*=.707;}
  if(dx)p.dir=dx>0?1:-1;
  if(!blocked(p.x+dx,p.y,PR))p.x+=dx;
  if(!blocked(p.x,p.y+dy,PR))p.y+=dy;
  const spd=Math.min(1.5+elapsed*.04,4);
  const ddx=p.x-l.x,ddy=p.y-l.y;
  const dist=Math.sqrt(ddx*ddx+ddy*ddy);
  if(dist>1){
    const mx=(ddx/dist)*spd,my=(ddy/dist)*spd;
    if(!blocked(l.x+mx,l.y+my,LR)){l.x+=mx;l.y+=my;}
    else{if(!blocked(l.x+mx,l.y,LR))l.x+=mx;if(!blocked(l.x,l.y+my,LR))l.y+=my;}
    l.dir=ddx>=0?1:-1;
  }
  if(dist<PR+LR+2){over=true;if(elapsed>best)best=elapsed;}
};
const render=()=>{
  ctx.clearRect(0,0,W,H);
  drawCourt();drawObs();
  drawBall(p.x+p.dir*20,p.y-10);
  drawFigure(p.x,p.y,'#e53935','11',p.dir);
  drawFigure(l.x,l.y,'#552583','23',l.dir);
  ctx.fillStyle='rgba(0,0,0,.72)';
  ctx.beginPath();ctx.roundRect(W/2-130,6,260,32,7);ctx.fill();
  ctx.textAlign='center';ctx.textBaseline='middle';
  ctx.fillStyle='#fff';ctx.font=`bold ${Math.round(W*.017)}px monospace`;
  ctx.fillText(`⏱ ${elapsed.toFixed(1)}s`,W/2,22);
  ctx.fillStyle='#fdb927';ctx.font='bold 11px monospace';
  ctx.textAlign='right';ctx.fillText(`Best: ${best.toFixed(1)}s`,W/2+124,22);
  const spd=Math.min(1.5+elapsed*.04,4);
  if(spd>2.8){
    ctx.fillStyle=`rgba(230,50,50,${Math.min((spd-2.8)*.5,.9)})`;
    ctx.font='bold 12px monospace';ctx.textAlign='center';
    ctx.fillText('⚡ LeBron is heating up!',W/2,46);
  }
  if(over){
    ctx.fillStyle='rgba(0,0,0,.75)';ctx.fillRect(0,0,W,H);
    const bw=Math.min(380,W*.75),bh=185,bx=(W-bw)/2,by=(H-bh)/2;
    ctx.fillStyle='#0d0d1a';ctx.strokeStyle='#fdb927';ctx.lineWidth=3;
    ctx.beginPath();ctx.roundRect(bx,by,bw,bh,14);ctx.fill();ctx.stroke();
    ctx.fillStyle='#fdb927';ctx.beginPath();ctx.roundRect(bx,by,bw,6,[14,14,0,0]);ctx.fill();
    ctx.textAlign='center';ctx.textBaseline='middle';
    ctx.fillStyle='#fdb927';ctx.font=`bold ${Math.round(bw*.062)}px monospace`;
    ctx.fillText('🏀 LeBron stole the ball!',W/2,by+48);
    ctx.fillStyle='#ff6f00';ctx.font=`bold ${Math.round(bw*.048)}px monospace`;
    ctx.fillText(`Survived  ${elapsed.toFixed(1)}s`,W/2,by+96);
    ctx.fillStyle='#888';ctx.font=`${Math.round(bw*.033)}px monospace`;
    ctx.fillText(`Best: ${best.toFixed(1)}s`,W/2,by+136);
    ctx.fillStyle='#ccc';ctx.fillText('Click court then press R',W/2,by+165);
  }
};
(function loop(){update();render();requestAnimationFrame(loop);})();
})();
</script>

# Homework
---------------------------------------------
## Iteration
---------------------------------------------
### Hack 1: The "Counting Stars" Hack (The while Loop) and Hack 2: The "Playlist Shuffle" Hack (The For Loop) 🎵

In [ ]:
// Hack 1
%%javascript

function countStars(limit) {
  // 1. Initialize a counter variable. We'll start our count at 1.
  let count = 1;


  // 2. Start the `while` loop. The code inside this loop will run
  // as long as the condition `count <= limit` is true.
  while (count <= limit) {
    // 3. This is the code that runs in each iteration.
    console.log(`Star #${count} shining bright!`);


    // 4. This is the increment step. It's crucial! If we don't
    // increase the `count`, the loop will run forever (an infinite loop).
    count += 1;
  }


  // 5. This message is displayed once the loop condition is no longer true
  // and the loop has finished.
  console.log("All the stars have been counted.");
}
countStars(5)

// Hack 2

%%javascript
function playlistShuffle(playlist) {
  // 1. Start the `for...of` loop. This loop is perfect for iterating
  // over the items of an array.
  // The variable `song` will hold the value of each item in the `playlist` array, one by one.
  for (const song of playlist) {
    // 2. Action performed for each song in the array.
    console.log(`Now playing: "${song}"`);
  }


  // 3. This message is displayed once the loop has processed every item in the array.
  console.log("All songs have been shuffled.");
}


// Example of how to define the array:
const mySongs = ["Heartless", "SKITZO", "Stargazing", "Too Many Nights", "one one tonight"]; 
// Then, call the function: playlistShuffle(mySongs);
playlistShuffle(mySongs);

## Conditionals

### <u>Popcorn Hack 1 - Simple Conditionals</u>

Instructions:
Finish the code in the cell below (fill in underlines and add lines of code) so it becomes a functioning conditional. What you make is up to you!

**Examples of conditionals you could use:**

1) Define your mood today and give yourself ice cream if you're happy
- challenge 1: add "else if" conditionals that echo different strings based on different moods
- challenge 2: add user input to define a "mood" variable

2) Define the weather today and echo "It's sunny outside!" if it's sunny
- challenge 1: add "else if" conditionals that define other weather
- challenge 2: let the user input today’s weather and respond automatically

3) Define your age as a variable and check whether you are an adult or a child.
- challenge 1: expand to add other categories using "else if" conditionals
- challenge 2: create a random age generator and test the conditional multiple times to see different results

The code is currently NOT functional. When you have completed this popcorn hack, the code should work. Have fun!

### <u>Popcorn Hack 2 - Vending Machine</u>

Instructions:
This hack is a bit more complicated than the first. Points will be awarded based on effort and completion.

Analyze the block of code below. Then, complete the following instructions:

You're hungry, so you go to your nearest vending machine for a snack. It makes you wonder if you'd be able to build your own. You can!... Digitally. Below, fill in the blanks so that when someone is prompted with food options, there are sub options within each food. (ex: Chocolate > Milk Chocolate)

1) **Create variables:**
- snackChoice -> user input (prompt) to determine which random snack is chosen (number from 1-3)

2) **Add basic conditionals determining what type of snack was chosen**

3) **Add nested if statements for extra behavior**

- Inside your chocolate condition:
    - Add user input to determine what type of chocolate the user wants
- Inside your candy condition:
    - Add user input to determine what color candy the user wants

4) **Challenge (OPTIONAL):**

- Make it so that there is a 10% chance of the vending machine giving you an extra cookie!

In [ ]:
// Hack 1
%%html

<div id="outputPH1">Results will appear here:<br></div>

<script>

    (() => {

        const output = document.getElementById("outputPH1");

        // Define a variable and assign it a value
        let mood = "happy";

        if (mood === "happy") {
            console.log("You’re happy! Enjoy some ice cream");
            output.innerText += "You’re happy! Enjoy some ice cream";
        } else if (mood === "sad") {
            console.log("Cheer up! Play basketball");
            output.innerText += "Cheer up! Play basketball";
        } else if (mood === "tired") {
            console.log("Take a nap");
            output.innerText += "Take a nap";
        } else {
            console.log("Mood not recognized, hope your day is going well!");
            output.innerText += "Mood not recognized, hope your day is going well!";
        }

    })();

</script>

// Hack 2

%%html

<div id="outputPH2">Results will appear here:<br></div>

<script>

(() => {

    const output = document.getElementById("outputPH2");

    let snackChoice = prompt("Choose a number: 1, 2, or 3 to get a snack.");

    if (snackChoice == "1") { // Chocolate bar
        let chocolateType = prompt("You got a chocolate bar! Choose your type: milk or dark?");
        
        // Nested condition for chocolate type
        if (chocolateType.toLowerCase() === "milk") { 
            output.innerText += "You chose a milk chocolate bar"; 
        } else if (chocolateType.toLowerCase() === "dark") { 
            output.innerText += "You chose a dark chocolate bar"; 
        } else {
            output.innerText += "That type of chocolate isn’t available for purchase"; 
        }

    } else if (snackChoice == "2") { // Chips
        output.innerText += "You chose a bag of chips"; 

    } else if (snackChoice == "3") { // Candy
        let candyColor = prompt("You got candy! Choose your color: red, blue, or yellow?");
        
        // Nested condition for candy color
        if (candyColor.toLowerCase() === "red") { 
            output.innerText += "You chose red cherry candy"; 
        } else if (candyColor.toLowerCase() === "blue") { 
            output.innerText += "You chose blue raspberry candy"; 
        } else if (candyColor.toLowerCase() === "yellow") { 
            output.innerText += "You chose lemon candy"; 
        } else {
            output.innerText += "That candy color isn’t available for purchase"; 
        }

    } else {
        output.innerText += "Invalid choice! Please choose 1, 2, or 3."; 
    }

    // Challenge: 10% chance of extra cookie
    if (Math.random() < 0.1) { // 10% chance
        output.innerText += "It's your lucky day! The machine gives you an extra cookie!";
    }

})();

</script>

## Arrays


In [ ]:
// Starting with a food list
let favoriteFoods = ["pizza", "ice cream", "burgers", "tacos", "pasta", "chocolate"];
console.log(favoriteFoods); // ["pizza", "ice cream", "burgers", "tacos", "pasta", "chocolate"]

// Adding "waffles" to the beginning
favoriteFoods.unshift("waffles");

// Result
console.log(favoriteFoods); // ["waffles", "pizza", "ice cream", "burgers", "tacos", "pasta", "chocolate"]

## Variables

In [ ]:
// Hack 1
%%javascript

// Step 1: Make some variables
let name = "Lance";
let age = 15;

// Step 2: Print a message
console.log("Hi, my name is", name + ".");
console.log("I am", age, "years old.");

// Step 3: Unfinished part
// TODO: Make a new variable called "nextYearAge"
// that is the age plus 1
let nextYearAge = age + 1;


// TODO: Print out the result
// Example: "Next year I will be 11 years old."
// console.log( ??? )   // <-- finish this line!
console.log("Next year I will be", nextYearAge, "years old.");


// Hack 2

%%javascript 

// Step 1: Define the Animal class
class Animal {
  // Initialize each animal with a name, sound, and type
  constructor(name, sound, kind) {
    this.name = name;
    this.sound = sound;
    this.kind = kind;
  }

  // animal speak
  speak() {
    console.log(`${this.name} the ${this.kind} says ${this.sound}!`);
  }

  // Bonus method: describe the animal
  describe() {
    console.log(`${this.name} is a ${this.kind} and is very friendly!`);
  }
}

// Step 2: Create a list to hold all the animals in the zoo
let zoo = [];

// Step 3: Add animals to the zoo
// TODO: Create at least 3 animals and push them into the zoo array
zoo.push(new Animal("Rocko", "Woof", "Dog"));
zoo.push(new Animal("Cam", "Meow", "Cat"));
zoo.push(new Animal("Perry", "Squawk", "Parrot"));

// Step 4: Loop through all animals and make them speak
// TODO: Use a for loop (or forEach) to call speak() on each animal
zoo.forEach(animal => {
  animal.speak();
  // Step 5: Optional bonus: Call describe() too
  animal.describe();
});

// Step 5 Bonus: Let the user add a new animal (works in browser)
// Uncomment if running in browser with prompt()
let name = prompt("Enter the animal's name:");
let sound = prompt("Enter the sound it makes:");
let kind = prompt("Enter the kind of animal:");
let newAnimal = new Animal(name, sound, kind);
zoo.push(newAnimal);
newAnimal.speak();
newAnimal.describe();

## Functions

In [ ]:
%%html

<p>Fahrenheit to Celsius Converter</p>

<input type="number" id="inputField" placeholder="Enter degrees (F)">
<button id="calculateButton">Calculate</button>

<div id="output1">Your results will appear here.</div>

<script>

(() => {

  const button = document.getElementById("calculateButton");
  const input = document.getElementById("inputField");
  const output = document.getElementById("output1")

  // ^^ DO NOT MODIFY ANY ABOVE CODE ^^

  // Define your function out here
  const toCelsius = f => (f - 32) / 1.8;

  button.addEventListener("click", () => {

    // Call your toCelsius() function, using the user's input from input.value.
    // Save the result to a variable.
    
    // Note that input.value returns a string. You will need to turn it into a
    // number using Number(), and then feed that number to your function.
  
    const result = toCelsius(Number(input.value));

    // Log the result to console
    console.log(result);

    // Add the result to the end of DOM:
    output.innerText += `\n${input.value}°F = ${result.toFixed(2)}°C`;

  // vv DO NOT MODIFY ANY BELOW CODE vv
  });

})();

// Hack 2

%%html

<div id="phack2">Your results will appear here:</div>

<script>

    (() => {

        const output = document.getElementById("phack2");

        // ^^ DO NOT MODIFY ABOVE CODE ^^

        const findMax = arr => {
            const max = Math.max(...arr);
            console.log("Max value:", max);
            output.innerText += `\nMax value: ${max}`;
            return max;
        };

        // Call the function three times with the given arrays
        findMax([1, 2, 3, 4, 4, 4, 5, 6, 8, 4, 1, 7]);
        findMax([1]);
        findMax([97883, 981, 81283, 987213, 1, -391]);

        // vv DO NOT MODIFY BELOW CODE vv

    })();
</script>

## Booleans

In [ ]:
%% html

<html>
<body>
  <h3>Boolean Challenge: Login System Requirements</h3>
  <div id="output"></div>

  <script>
  (() => {
    // Challenge: Write boolean expressions for these conditions
var hasDecimal = false;       // true if current number already has a decimal
var isNegative = false;       // true if current number is negative
var justCalculated = false;   // true right after you press "="

var hasDecimal = false;


## Variables

In [ ]:
%%html

<html>
<body>
  <h2>Popcorn Hack 1 Output</h2>
  <div id="output1"></div>

  <script>
  (() => {

    let name = "Alex";
    const age = 2 ;
    let hobby = "painting";

    // In practice, you shouldn't use var. This is just for the purposes of teaching :)
    var city = "New York"; 

    // Change vars here
    name = "Bessie";
    //age = 2;   // What happens if you uncomment this line ???
    city = "Anishiapolis";

    document.getElementById("output1").innerText = name + " is " + age + " years old and lives in " + city + " and loves " + hobby + ".";
    console.log(name + " is " + age + " years old and lives in " + city + " and loves " + hobby + ".")

  })();
  </script>
</body>
</html>


## Mathematical Expressions

In [ ]:
%%html

<!-- This is where the outputs of your functions will go: -->
<h2>Function outputs:</h2>


<!-- Modify each of the button onclick fields to call your functions: -->
<!-- Make sure when the function is called you input your parameters. -->
<!-- (Optional) You can also change the text each button displays if you want. -->
<button type="button" onclick="Add(4,5)">Function Output 1</button>
<div id="functionOutput1"></div>
<br>

<button type="button" onclick="Subtraction(10,5)">Function Output 2</button>
<div id="functionOutput2"></div>
<br>

<button type="button" onclick="Multiplication(4,5)">Function Output 3</button>
<div id="functionOutput3"></div>
<br>

<button type="button" onclick="Exponential(4,5)">Function Output 4</button>
<div id="functionOutput4"></div>
<br>

<button type="button" onclick="Modulus(89,5)">Function Output 5</button>
<div id="functionOutput5"></div>
<br>

<button type="button" onclick="squareRoot(25)">Function Output 6</button>
<div id="functionOutput6"></div>
<br>

<button type="button" onclick="squareRoot(25)">Function Output 6</button>
<div id="functionOutput6"></div>
<br>

<button type="button" onclick="Division(10,5)">Function Output 7</button>
<div id="functionOutput7"></div>
<br>

<button type="button" onclick="cubedRoot(27)">Function Output 8</button>
<div id="functionOutput8"></div>
<br>

<button type="button" onclick="Cubed(3)">Function Output 9</button>
<div id="functionOutput9"></div>
<br>


<!-- You will write the code for your functions in this script area  -->
<script>
	// Addition
	function Add(a,b)
	{
		let result = a + b;
		
		// This line is important. It is what changes the output text section to the result of the function
		document.getElementById("functionOutput1").innerText = result;
	}

	// Subtraction
	function Subtraction(a,b) 
	{
		let result = a - b;
		document.getElementById("functionOutput2").innerText = result;
	}

	// Multiplication
	function Multiplication(a,b) 
	{
		let result = a * b;
		document.getElementById("functionOutput3").innerText = result;
	}

	// Exponential
	function Exponential(a,b) 
	{
		let result = a ** b;
		document.getElementById("functionOutput4").innerText = result;
	}
	// Modulus
	function Modulus(a,b) 
	{
		let result = a % b;
		document.getElementById("functionOutput5").innerText = result;
	}

	// Add all of your own functions here:
	
	// Square Root
	function squareRoot(a) 
	{
		let result = Math.sqrt(a);
		document.getElementById("functionOutput6").innerText = result;
	}
	
	// Division
	function Division(a,b) 
	{
		let result = a / b;
		document.getElementById("functionOutput7").innerText = result;
	}

	// Cubed Root
	function cubedRoot(a) 
	{
		let result = Math.cbrt(a);
		document.getElementById("functionOutput8").innerText = result;
	}

	// Cubed
	function Cubed(a) 
	{
		let result = a * a * a;
		document.getElementById("functionOutput9").innerText = result;
	}

	// You can make them do any mathematical expression, just make sure they change the text of one of the output sections
</script>



## Classes and Methods

In [ ]:
%%javascript

class Dice { // What should the name of the class be?
  constructor(sides) { // Constructor to initialize the number of sides
    this.sides = sides; // Number of sides on the dice
  }

  // ____() { // What should the name of the method be?
  roll() { 
    return Math.floor(Math.random() * this.sides) + 1; // Rolls the dice and returns a number between 1 and the number of sides
  }

// Test the Dice class:

// INSTANTIATE A NEW DICE OBJECT WITH A NUMBER OF SIDES:

// CALL THE ROLL METHOD A FEW TIMES AND PRINT THE RESULTS
let myDice = new Dice(6);
console.log("Roll 1:", myDice.roll());
console.log("Roll 2:", myDice.roll());
console.log("Roll 3:", myDice.roll());

## Arrays

In [ ]:
%%html

<html>

<body>
  <h2>Output 2</h2>
  <div id="outputpls"></div>

  <script>
    document.getElementById("outputpls").innerText = ""; // Clear output
    (() => {

        let titleofthings = ["Play basketball", " play videogames", " and hang out with friends", ];
        const sentence = "I like to: " + titleofthings;
        console.log(sentence);
        document.getElementById("outputpls").innerText += sentence;


    })();
  </script>
</body>
</html>

## Data Abstraction

In [ ]:
%%javascript
let username = "LeBron_James";
let password = "imking23";

// Part 1: Print username and password
function Account() {
    console.log("Username:", username);
    console.log("Password:", password);
}

// Part 2: Change the username
function ChangeUsername(newUsername) {
    username = newUsername;
    console.log("Username changed to:", username);
}

// Part 3: Change the password
function ChangePassword(newPassword) {
    password = newPassword;
    console.log("Password changed to:", password);
}

// Part 4: Login check
function Login(_username, _password) {
    if (_username === username && _password === password) {
        console.log("Login successful!");
        return true;
    } else {
        console.log("Login failed!");
        return false;
    }
}

// --- CALL FUNCTIONS TO PRODUCE OUTPUT ---
Account();                     // Show current credentials
ChangeUsername("Lebonbon");  // Update username
ChangePassword("imtuff23"); // Update password
Login("Lebonbon", "imtuff23"); // Correct login
Login("Lebonbon", "wrong");    // Failed login

## Strings

In [ ]:
// Task 1: Define your variables and strings

let _____ = "______ "; 
let ______ = "______";
let _______ = "______";

// Task 2: Write the complex sentence using a template literal (backticks).
// Use ${} to insert the variables directly.
const snackMessage = `

`;

// Task 3. Check the length of the full message and print it to the console.

console.log(`The full message is ${snackMessage.length} characters long.`);

console.log(snackMessage);

## Reflection

Building this game was the best way to actually understand these CS111 concepts because I had to use all of them together at once. The `for` loop draws the court, the `Array` holds the obstacles, the `Object` tracks player state, `Boolean` flags control game flow, and the math operators power the physics and AI. Seeing how they all connect in a real project made it way easier to understand than just reading about them separately.